# Signatures comparison — Rare cell types (CHESS-1336)

Companion to `Signatures comparison VALIDATION.ipynb` (its §11 spec'd this notebook).
The four FGES whose GOI is rare in the new OD-128 validation cohort —
`Main4_Th17_signature`, `Main4_Lymphatic_endothelium`, `Main4_Eosinophil_signature`,
`Main4_Plasma_cells` (`EXCLUDED_FGES_RARE`) — are scored here on the **original**
(v1) cohort instead, whose per-sample ssGSEA scores are already persisted in
`data/mapping_ssgseas.pkl`.

**Why reuse precomputed scores instead of re-running ssGSEA on raw expressions:**
`ssgsea_formula` ranks each sample's genes independently (`expressions.rank()` is
per-column, i.e. per sample), so ssGSEA is a strictly single-sample method — a
sub-signature's score for sample X never depends on which other samples are scored
alongside it. Slicing the full-cohort score matrix down to a test-fold's sample IDs
is therefore identical to re-running ssGSEA on that fold alone.

**Method (rare-types rerun):** for each of the 4 FGES, 10 independent 75/25
stratified holdouts are drawn via
`signature_validation.benchmark.splits.stratified_holdout_indices`, stratified by
(BG-FGES score above/below median) × (GOI/Control). Each holdout's test-fold scores
are collected and reduced across folds via `aggregate_score_over_splits` (mean per
sample, over however many of the 10 folds happened to include it).

**Outputs:** written under `../plots` and `../tables` with a `_rare` suffix; v1 /
NEW / VALIDATION outputs are never overwritten. All figures star every FGES and
every GOI/control cell type here (all four FGES are members of
`EXCLUDED_FGES_RARE`, and their GOI cell types are exactly `RARE_CELL_TYPES`).

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from loguru import logger

from signature_validation.benchmark.cohorts import (
    CONTROLS_ORDER,
    EXCLUDED_FGES_RARE,
    intersect_controls_with_cohort,
)
from signature_validation.benchmark.plotting import (
    RARE_CELL_TYPES,
    RARE_FGES_KEYS,
    plot_sens_spec_scatter,
    plot_signature_heatmap,
    plot_violin_per_source,
)
from signature_validation.benchmark.scoring import compute_out_table, fdr_correct_out
from signature_validation.benchmark.signatures import (
    count_random_fges,
    load_v1_msigdb_gmt,
    select_msigdb_gmt_subset,
)
from signature_validation.benchmark.splits import (
    aggregate_score_over_splits,
    stratified_holdout_indices,
)
from signature_validation.plotting.plotting import cells_p
from signature_validation.utils.utils import RANDOM_SEED

sns.set_style("white")
plt.rcParams["svg.fonttype"] = "none"

In [ ]:
import bioreactor

## §1 — Inputs and outputs

In [ ]:
# --- Original-cohort inputs (real local pickles, committed under data/) ---
ORIGINAL_SSGSEAS_PATH = Path("../data/mapping_ssgseas.pkl")  # v1 per-sample ssGSEA scores, 199 MB
V1_GMT_PICKLE = Path("../data/msigdb_gmt.pkl")               # v1 gene lists (sub-signature names only needed here)

# --- Outputs (repo folders, `_rare` suffix; v1/NEW/VALIDATION never overwritten) ---
OUTPUT_DIR = Path("../plots")
TABLES_DIR = Path("../tables")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

MAPPING_SSGSEAS_RARE_PATH = OUTPUT_DIR / "mapping_ssgseas_rare.pkl"
OUT_TSV_RARE_PATH = TABLES_DIR / "out_rare.tsv"
HEATMAP_RARE_PATH = OUTPUT_DIR / "signature_heatmap_rare.svg"

N_SPLITS = 10
TEST_SIZE = 0.25

for label, p in (
    ("original-cohort ssGSEA pickle", ORIGINAL_SSGSEAS_PATH),
    ("v1 GMT pickle", V1_GMT_PICKLE),
):
    if not p.exists():
        logger.error("{} not found: {}", label, p)

logger.info("outputs -> {} and {}", OUTPUT_DIR, TABLES_DIR)

## §2 — Load the original cohort's ssGSEA scores and v1 gene lists

In [ ]:
if not ORIGINAL_SSGSEAS_PATH.exists():
    raise FileNotFoundError(f"original-cohort scores not found: {ORIGINAL_SSGSEAS_PATH}")
with open(ORIGINAL_SSGSEAS_PATH, "rb") as fh:
    head = fh.read(64)
if head.startswith(b"version https://git-lfs"):
    raise RuntimeError(
        f"{ORIGINAL_SSGSEAS_PATH} is an unresolved git-LFS pointer; run `git lfs pull`"
    )
with open(ORIGINAL_SSGSEAS_PATH, "rb") as fh:
    original_ssgseas = pickle.load(fh)
logger.info("original-cohort scores loaded: {} FGES", len(original_ssgseas))

v1_gmt_full = load_v1_msigdb_gmt(V1_GMT_PICKLE)
rare_gmt = select_msigdb_gmt_subset(v1_gmt_full, sorted(EXCLUDED_FGES_RARE))

for sign in sorted(EXCLUDED_FGES_RARE):
    assert sign in original_ssgseas, f"{sign}: missing from original-cohort scores"
    assert sign in rare_gmt[sign], f"v1 GMT[{sign}] missing the BG sub-signature"
    n_random = count_random_fges(rare_gmt[sign])
    assert n_random == 10, f"{sign}: expected 10 RANDOM_FGES, got {n_random}"
logger.info("rare_gmt: {} FGES, {} sub-signatures total", len(rare_gmt), sum(len(v) for v in rare_gmt.values()))

## §3 — Stratified 75/25 holdouts (x10) and cross-fold aggregation

Per FGES: GOI + Control samples (as already scored for the original cohort, i.e.
`original_ssgseas[sign]["Goi"|"Control"]`) are pooled, stratified on
(BG-FGES score >= median) × (GOI/Control), and split into 10 independent 75/25
holdouts via `stratified_holdout_indices`. Each split's test-fold scores are the
precomputed scores sliced to that fold's sample ids (see §0 — ssGSEA is
single-sample, so this *is* "scoring on the test fold"). `aggregate_score_over_splits`
then averages, per sample, over whichever of the 10 test folds happened to include
it. `Deleted_controls` are outside the GOI/Control comparison entirely and are
carried over unresampled, for heatmap completeness only.

In [ ]:
def _label_group(group_frames: dict[str, pd.DataFrame], label: str) -> tuple[pd.Series, pd.Series]:
    """Sample-indexed (cell_type, cohort_label) Series for one Goi/Control bucket."""
    ct_parts, label_parts = [], []
    for ct, df in group_frames.items():
        if df.empty:
            continue
        ct_parts.append(pd.Series(ct, index=df.index))
        label_parts.append(pd.Series(label, index=df.index))
    if not ct_parts:
        return pd.Series(dtype=object), pd.Series(dtype=object)
    return pd.concat(ct_parts), pd.concat(label_parts)


def build_rare_fold_aggregate(
    sign: str,
    original_ssgseas: dict,
    n_splits: int = N_SPLITS,
    test_size: float = TEST_SIZE,
    random_state: int = RANDOM_SEED,
) -> tuple[dict[str, dict[str, pd.DataFrame]], list[pd.Index]]:
    """10x stratified 75/25 holdout rerun for one rare FGES (CHESS-1336 §11)."""
    goi_frames = original_ssgseas[sign]["Goi"]
    ctrl_frames = original_ssgseas[sign]["Control"]

    goi_ct, goi_label = _label_group(goi_frames, "GOI")
    ctrl_ct, ctrl_label = _label_group(ctrl_frames, "Control")
    cohort_label = pd.concat([goi_label, ctrl_label])
    cohort_label = cohort_label[~cohort_label.index.duplicated(keep="first")]

    full_df = pd.concat(list(goi_frames.values()) + list(ctrl_frames.values()))
    full_df = full_df[~full_df.index.duplicated(keep="first")]
    bg_score = full_df[sign]

    splits = stratified_holdout_indices(
        samples=full_df.index.tolist(),
        score=bg_score,
        cohort_label=cohort_label,
        n_splits=n_splits,
        test_size=test_size,
        random_state=random_state,
    )
    per_split_scores = [full_df.loc[idx] for idx in splits]
    aggregated = aggregate_score_over_splits(per_split_scores, how="mean")

    n_goi = int((cohort_label == "GOI").sum())
    logger.info(
        "{}: {} GOI / {} Control samples -> {} splits ({:.0%} test) -> {} samples covered by >=1 test fold",
        sign, n_goi, len(cohort_label) - n_goi, n_splits, test_size, len(aggregated),
    )

    rerun: dict[str, dict[str, pd.DataFrame]] = {"Goi": {}, "Control": {}, "Deleted_controls": {}}
    for ct, df in goi_frames.items():
        kept = aggregated.index.intersection(df.index)
        if len(kept):
            rerun["Goi"][ct] = aggregated.loc[kept]
    for ct, df in ctrl_frames.items():
        kept = aggregated.index.intersection(df.index)
        if len(kept):
            rerun["Control"][ct] = aggregated.loc[kept]
    rerun["Deleted_controls"] = dict(original_ssgseas[sign]["Deleted_controls"])
    return rerun, splits


mapping_ssgseas_rare: dict = {}
mapping_rare: dict = {}
splits_used: dict = {}
for sign in sorted(EXCLUDED_FGES_RARE):
    rerun, splits = build_rare_fold_aggregate(sign, original_ssgseas)
    mapping_ssgseas_rare[sign] = rerun
    mapping_rare[sign] = {
        "Goi": list(original_ssgseas[sign]["Goi"].keys()),
        "Control": list(original_ssgseas[sign]["Control"].keys()),
        "Deleted_controls": list(original_ssgseas[sign]["Deleted_controls"].keys()),
    }
    splits_used[sign] = splits

with open(MAPPING_SSGSEAS_RARE_PATH, "wb") as fh:
    pickle.dump(mapping_ssgseas_rare, fh, pickle.HIGHEST_PROTOCOL)
logger.info("wrote {}", MAPPING_SSGSEAS_RARE_PATH)

## §4 — Per-signature × cell-type stats table (+ deterministic FDR)

In [ ]:
sample_cell_type_all = pd.concat(
    [
        pd.Series(ct, index=df.index)
        for sign in mapping_rare
        for group in ("Goi", "Control", "Deleted_controls")
        for ct, df in original_ssgseas[sign][group].items()
        if not df.empty
    ]
)
sample_cell_type_all = sample_cell_type_all[~sample_cell_type_all.index.duplicated(keep="first")]
annotation_rare = pd.DataFrame({"Cell_type": sample_cell_type_all})

controls_present_rare = intersect_controls_with_cohort(CONTROLS_ORDER, annotation_rare)
logger.info("controls present across the 4 rare FGES: {}", len(controls_present_rare))

out_rare = compute_out_table(mapping_ssgseas_rare, mapping_rare, rare_gmt, controls_present_rare)
out_rare = fdr_correct_out(out_rare, controls_present_rare)
out_rare.to_csv(OUT_TSV_RARE_PATH, sep="\t")
logger.info("wrote {} ({} rows x {} cols)", OUT_TSV_RARE_PATH, *out_rare.shape)
out_rare.head()

## §5 — Figures (suffix `_rare`; every FGES / GOI cell type starred)

All four FGES scored here are members of `EXCLUDED_FGES_RARE`, and their GOI cell
types are exactly `RARE_CELL_TYPES` — so with the plot helpers' default
`rare_fges`/`rare_cell_types` arguments, every row in these figures is starred.

In [ ]:
plot_violin_per_source(mapping_ssgseas_rare, save_dir=OUTPUT_DIR, suffix="_rare")

plot_signature_heatmap(
    mapping_ssgseas=mapping_ssgseas_rare,
    out_df=out_rare,
    mapping=mapping_rare,
    msigdb_gmt=rare_gmt,
    annotation=annotation_rare,
    controls_order=controls_present_rare,
    palette={ct: cells_p[ct] for ct in controls_present_rare if ct in cells_p},
    save_path=HEATMAP_RARE_PATH,
    short=True,
)

rare_avg = plot_sens_spec_scatter(
    mapping_ssgseas=mapping_ssgseas_rare,
    msigdb_gmt=rare_gmt,
    mapping=mapping_rare,
    save_dir=OUTPUT_DIR,
    suffix="_rare",
)
logger.info("plots saved under {}", OUTPUT_DIR)